"""
=========================================================================
 MODELLING THE EFFECTS OF LARGE INFRASTRUCTURE ON URBAN TRAFFIC FLOW DURING CONSTRUCTION
=========================================================================
 Paper: "Second-Order Macroscopic Traffic Model with Unified Construction
         Activity Field for Work-Zone Traffic Management"

 GOVERNING EQUATIONS
 -------------------
 (1) Continuity:
       dk/dt + d(kv)/dx = 0

 (2) Dynamic Velocity:
       dv/dt + v*dv/dx = (ve(k;x,t) - v)/TC(x,t)
                       + c0*dv/dx
                       + mu0*d2v/dx2
                       - alpha*h(W)*v
                       - lambdaL*v*dL/dx

 CONSTRUCTION FIELDS
 -------------------
       h(W(x,t))       = W(x,t) + delta*chiD(x,t)        [unified activity]
       vf(x,t)       = vf0*(1 - etaW*W(x,t))           [free-flow speed]
       keff(x,t)     = k0*L(x,t)                        [effective jam density]
       TC(x,t)       = T0*(1 + tau_h*Wh(x,t))          [relaxation time]
       ve(k;x,t)     = vf*[1-exp(1-exp((cm/vf)*(keff/k - 1)))]_+
       L(x,t) smooth taper via tanh profile

 NUMERICAL SCHEME
 ----------------
       Continuity  : Conservative Lax-Friedrichs flux
       Advection   : Upwind (sign of v)
       Anticipation: Upwind (sign of v - c0)
       Diffusion   : Central second-order
       CFL         : dt <= dx/(vf0 + c0)
"""

In [1]:
import numpy as np
import matplotlib
matplotlib.use('TkAgg')   
import matplotlib.pyplot as plt
from matplotlib.cm import get_cmap

np.random.seed(42)

In [2]:
#  SECTION 1: PARAMETERS

p = {}

# Spatial domain 
p['L_road'] = 4000.0          
p['Nx']     = 400              
p['dx']     = p['L_road'] / p['Nx']   
p['x']      = np.arange(0.5*p['dx'], p['L_road'], p['dx'])  

# Time domain
p['T_end'] = 900.0             
p['dt']    = 0.25              
p['Nt']    = round(p['T_end'] / p['dt'])

# Baseline traffic parameters
p['vf0'] = 28.0    # [m/s]    Free-Flow speed  
p['k0']  = 0.14    # [veh/m]  Baseline Jam Density
p['cm']  = 11.0    # [m/s]    Kinematic wave speed at capacity
p['T0']  = 4.0     # [s]      Baseline Relaxation Time
p['c0']  = 6.0     # [m/s]    Anticipation Coefficient
p['mu0'] = 12.0    # [m2/s]   Diffusion Coefficient

# Construction parameters
p['etaW']    = 0.45  # [-]      free-speed suppression p
p['tau_h']   = 1.80  # [-]      hesitation penalty 
p['delta']   = 0.35  # [-]      detour weight in unified activity
p['alpha']   = 0.10  # [1/s]    unified drag term
p['lambdaL'] = 8.0   # [m/s]    anticipatory braking sensitivity

#Taper geometry
p['ell'] = 150.0   #     taper length for smooth

# Physical bounds
p['k_min'] = 1e-4  # [veh/m]
p['v_min'] = 0.01  # [m/s]
p['v_max'] = p['vf0']

# CFL condition
v_max_char = p['vf0'] + p['c0']
cfl = v_max_char * p['dt'] / p['dx']
print(f"CFL = {cfl:.4f}  (must be < 1)")
if cfl >= 1.0:
    p['dt'] = 0.85 * p['dx'] / v_max_char
    p['Nt'] = round(p['T_end'] / p['dt'])
    print(f"  dt auto-adjusted to {p['dt']:.4f} s")

# Diffusion stability 
if p['dt'] > p['dx']**2 / (2 * p['mu0']):
    print("  WARNING: Diffusion stability marginal. Consider reducing mu0.")

print(f"Grid: dx={p['dx']:.1f} m, dt={p['dt']:.3f} s, Nt={p['Nt']}\n")


CFL = 0.8500  (must be < 1)
Grid: dx=10.0 m, dt=0.250 s, Nt=3600



In [3]:
#  LOCAL FUNCTIONS

def compute_ve(k, vf, keff, cm):
    """Del Castillo-Benitez equilibrium speed, construction-aware.
    ve(k;x,t) = vf * [1 - exp(1 - exp((cm/vf)*(keff/k - 1)))]_+
    """
    k    = np.maximum(k,    1e-5)
    keff = np.maximum(keff, 1e-5)
    vf   = np.maximum(vf,   0.1)
    arg  = (cm / vf) * (keff / k - 1.0)
    return vf * np.maximum(0.0, 1.0 - np.exp(1.0 - np.exp(arg)))


def compute_ve_simple(k, p):
    """Baseline equilibrium (no construction fields)."""
    k = np.atleast_1d(np.asarray(k, dtype=float))
    return compute_ve(k, p['vf0'] * np.ones_like(k),
                         p['k0']  * np.ones_like(k), p['cm'])


def smooth_lane_field(x, x0, L_before, L_after, ell):
    """Single tanh taper from L_before to L_after at x0.
    L(x) = L_before + (L_after - L_before)/2 * (1 + tanh((x - x0)/ell))
    """
    return L_before + 0.5 * (L_after - L_before) * (1.0 + np.tanh((x - x0) / ell))


def analytical_dL_dx_single(x, x0, L_before, L_after, ell):
    """Analytical gradient of a single tanh taper (eq. 3.4.52).
    dL/dx = (L_after - L_before) / (2*ell) * sech^2((x - x0) / ell)
    sech^2(u) = 1 - tanh^2(u)
    """
    u = (x - x0) / ell
    sech2 = 1.0 - np.tanh(u) ** 2          # sech²(u) = 1 − tanh²(u)
    return (L_after - L_before) / (2.0 * ell) * sech2


def smooth_lane_field_double(x, x_in, x_out, L_b, L_a, L_after, ell):
    """Entry taper at x_in, exit taper at x_out."""
    L_entry = L_b + 0.5 * (L_a     - L_b)     * (1.0 + np.tanh((x - x_in)  / ell))
    L_exit  = L_a + 0.5 * (L_after - L_a)     * (1.0 + np.tanh((x - x_out) / ell))
    L_out   = L_entry.copy()
    mask    = x > x_out - 2 * ell
    L_out[mask] = L_exit[mask]
    return L_out


def analytical_dL_dx_double(x, x_in, x_out, L_b, L_a, L_after, ell):
    """Analytical gradient of a double tanh taper (eq. 3.4.52 applied piecewise).
    Entry region:  dL/dx = (L_a - L_b)   / (2*ell) * sech²((x - x_in)  / ell)
    Exit  region:  dL/dx = (L_after - L_a)/ (2*ell) * sech²((x - x_out) / ell)
    The same mask used to build L_field selects which formula applies.
    """
    
    u_in   = (x - x_in)  / ell
    sech2_in  = 1.0 - np.tanh(u_in)  ** 2
    dL_entry  = (L_a     - L_b)    / (2.0 * ell) * sech2_in

    
    u_out  = (x - x_out) / ell
    sech2_out = 1.0 - np.tanh(u_out) ** 2
    dL_exit   = (L_after - L_a)    / (2.0 * ell) * sech2_out

    dL_out = dL_entry.copy()
    mask   = x > x_out - 2 * ell
    dL_out[mask] = dL_exit[mask]
    return dL_out


def activity_gaussian(x, x_start, x_end, W_max):
    """Smooth bell-shaped work activity between x_start and x_end."""
    x_c   = 0.5 * (x_start + x_end)
    sigma = (x_end - x_start) / 4.0
    W_out = W_max * np.exp(-0.5 * ((x - x_c) / sigma) ** 2)
    W_out[x < x_start - 2 * sigma] = 0.0
    W_out[x > x_end   + 2 * sigma] = 0.0
    return np.clip(W_out, 0.0, 1.0)

In [4]:
#  SECTION 2: CONSTRUCTION FIELD

def build_construction_fields(scenario, t, p):
    """Return L(x), W(x), chiD(x), dL_dx(x) for a named scenario.

    dL_dx is computed analytically using eq. 3.4.52 from the paper:
        dL/dx = (L_after - L_before) / (2*ell) * sech²((x - x0) / ell)
    so that the code exactly mirrors the closed-form expression derived in
    the manuscript rather than using a numerical finite-difference gradient.
    """
    x  = p['x']
    Nx = p['Nx']

    L_field    = np.ones(Nx)
    W_field    = np.zeros(Nx)
    chiD_field = np.zeros(Nx)
    dL_field   = np.zeros(Nx)   

    x_zone_start = 1500.0
    x_zone_end   = 2800.0
    x0_taper_in  = x_zone_start + 200.0
    x0_taper_out = x_zone_end   - 200.0
    ell          = p['ell']

    if scenario == 'baseline':
        pass  # L=1 everywhere

    elif scenario == 'single_lane':
        L_before, L_after = 1.0, 2.0/3.0
        L_field  = smooth_lane_field_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        dL_field = analytical_dL_dx_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        W_field  = activity_gaussian(x, x_zone_start, x_zone_end, 0.65)

    elif scenario == 'double_lane':
        L_before, L_after = 1.0, 1.0/3.0
        L_field  = smooth_lane_field_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        dL_field = analytical_dL_dx_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        W_field  = activity_gaussian(x, x_zone_start, x_zone_end, 0.90)

    elif scenario == 'moving_zone':
        shift   = 0.5 * t
        x_c_in  = min(x0_taper_in  + shift, p['L_road'] - 400.0)
        x_c_out = min(x0_taper_out + shift, p['L_road'] - 200.0)
        L_before, L_after = 1.0, 2.0/3.0
        L_field  = smooth_lane_field_double(x, x_c_in, x_c_out,
                                            L_before, L_after, L_before, ell)
        dL_field = analytical_dL_dx_double(x, x_c_in, x_c_out,
                                            L_before, L_after, L_before, ell)
        W_field  = activity_gaussian(x, x_c_in - 200.0, x_c_out + 200.0, 0.70)

    elif scenario == 'detour':
        L_before, L_after = 1.0, 0.5
        L_field    = smooth_lane_field_double(x, x0_taper_in, x0_taper_out,
                                              L_before, L_after, L_before, ell)
        dL_field   = analytical_dL_dx_double(x, x0_taper_in, x0_taper_out,
                                              L_before, L_after, L_before, ell)
        W_field    = activity_gaussian(x, x_zone_start, x_zone_end, 0.75)
        chiD_field = activity_gaussian(x, x_zone_end, x_zone_end + 600.0, 0.60)

    elif scenario == 'night_works':
        t_peak  = 300.0
        sigma_t = 250.0
        W_amp   = np.exp(-0.5 * ((t - t_peak) / sigma_t) ** 2)
        L_before = 1.0
        L_after  = max(1.0/3.0, 1.0 - 0.67 * W_amp)
        L_field  = smooth_lane_field_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        dL_field = analytical_dL_dx_double(x, x0_taper_in, x0_taper_out,
                                            L_before, L_after, L_before, ell)
        W_field  = activity_gaussian(x, x_zone_start, x_zone_end, 0.90 * W_amp)

    else:
        raise ValueError(f"Unknown scenario: {scenario}")

    L_field    = np.clip(L_field,    0.05, 1.0)
    W_field    = np.clip(W_field,    0.0,  1.0)
    chiD_field = np.clip(chiD_field, 0.0,  1.0)
    return L_field, W_field, chiD_field, dL_field

In [5]:
#  SECTION 3: TIME-STEP UPDATE

def step_construction(k, v, scenario, t, p):
    """Advance density k and velocity v by one time step dt."""
    Nx = p['Nx']
    dx = p['dx']
    dt = p['dt']

    L, W, chiD, dL_dx = build_construction_fields(scenario, t, p)
    Wh   = W + p['delta'] * chiD
    vf   = np.maximum(0.1, p['vf0'] * (1.0 - p['etaW'] * W))
    keff = np.maximum(1e-3, p['k0'] * L)
    Tc   = p['T0'] * (1.0 + p['tau_h'] * Wh)
    ve   = compute_ve(k, vf, keff, p['cm'])


    # Lax-Friedrichs
    F = np.zeros(Nx + 1)
    F[0]  = k[0]  * v[0]
    F[-1] = k[-1] * v[-1]
    for i in range(1, Nx):
        uij  = 0.5 * (v[i-1] + v[i])
        F[i] = (0.5 * (k[i-1]*v[i-1] + k[i]*v[i])
                - 0.5 * abs(uij) * (k[i] - k[i-1]))

    k_new = k - (dt / dx) * (F[1:] - F[:-1])

    
    eps_k    = 0.01
    k_smooth = k_new.copy()
    k_smooth[1:-1] = (k_new[1:-1]
                      + eps_k * (k_new[2:] - 2.0*k_new[1:-1] + k_new[:-2]))
    k_new = k_smooth

    # Velocity equation terms
    dv_adv = np.zeros(Nx)
    dv_ant = np.zeros(Nx)
    d2v    = np.zeros(Nx)

    for i in range(1, Nx - 1):
        if v[i] >= 0:
            dv_adv[i] = (v[i] - v[i-1]) / dx
        else:
            dv_adv[i] = (v[i+1] - v[i]) / dx

        # Anticipation 
        a = v[i] - p['c0']
        if a >= 0:
            dv_ant[i] = (v[i] - v[i-1]) / dx
        else:
            dv_ant[i] = (v[i+1] - v[i]) / dx

        # Diffusion 
        d2v[i] = (v[i+1] - 2.0*v[i] + v[i-1]) / dx**2

    # Boundary
    dv_adv[0]   = (v[1]  - v[0])    / dx
    dv_adv[-1]  = (v[-1] - v[-2])   / dx
    dv_ant[0]   = dv_adv[0]
    dv_ant[-1]  = dv_adv[-1]
    d2v[0]      = d2v[1]
    d2v[-1]     = d2v[-2]

    
    # Construction drag:        -alpha * h(W)* v
    drag      = -p['alpha']   * Wh * v
    # Anticipatory braking:     -lambdaL * v * dL/dx  
    ant_brake = -p['lambdaL'] * v  * dL_dx

    # Velocity update 
    v_new = v + dt * (
        - v * dv_adv                  #  -v * dv/dx
        + (ve - v) / Tc               # Relaxation:      (Ve - v) / Tc
        + p['c0'] * dv_ant            # Anticipation:    c0 * dv/dx
        + p['mu0'] * d2v              # Diffusion:       mu0 * d²v/dx²
        + drag                        # Construction Drag
        + ant_brake                   # Anticipatory Braking
    )

    # Physical bounds
    k_new = np.clip(k_new, p['k_min'], p['k0'])
    v_new = np.clip(v_new, p['v_min'], p['v_max'])
    return k_new, v_new

In [6]:
#  SECTION 4: FULL SIMULATION

def run_sim(scenario, p, k_in_val, save_stride):
    """Run full simulation; return K, V, Q, Wh_field, t_vec arrays."""
    Nx = p['Nx']

    k    = k_in_val * np.ones(Nx)
    v    = compute_ve_simple(k, p)
    v_in = float(compute_ve_simple(np.array([k_in_val]), p)[0])

    Nt_save  = p['Nt'] // save_stride + 1
    K        = np.zeros((Nx, Nt_save))
    V        = np.zeros((Nx, Nt_save))
    Q        = np.zeros((Nx, Nt_save))
    Wh_field = np.zeros((Nx, Nt_save))
    t_vec    = np.zeros(Nt_save)

    idx = 0
    K[:, idx] = k
    V[:, idx] = v
    Q[:, idx] = k * v
    t_vec[idx] = 0.0
    _, W0, chiD0, _ = build_construction_fields(scenario, 0.0, p)
    Wh_field[:, idx] = W0 + p['delta'] * chiD0

    for n in range(1, p['Nt'] + 1):
        t_now = (n - 1) * p['dt']
        k_new, v_new = step_construction(k, v, scenario, t_now, p)

        # Boundary conditions
        k_new[0]  = k_in_val
        v_new[0]  = v_in
        k_new[-1] = k_new[-2]
        v_new[-1] = v_new[-2]

        k = k_new
        v = v_new

        if n % save_stride == 0:
            idx += 1
            if idx < Nt_save:
                K[:, idx]   = k
                V[:, idx]   = v
                Q[:, idx]   = k * v
                t_vec[idx]  = n * p['dt']
                _, Wn, cDn, _ = build_construction_fields(scenario, n * p['dt'], p)
                Wh_field[:, idx] = Wn + p['delta'] * cDn

    K        = K[:, :idx+1]
    V        = V[:, :idx+1]
    Q        = Q[:, :idx+1]
    Wh_field = Wh_field[:, :idx+1]
    t_vec    = t_vec[:idx+1]
    return K, V, Q, Wh_field, t_vec

In [7]:
#  SECTION 5: RUN ALL SCENARIOS

save_stride  = 4
scenarios    = ['baseline', 'single_lane', 'double_lane',
                'detour', 'night_works', 'moving_zone']
scen_labels  = ['Baseline (No Construction)', 'Single Lane Closure',
                'Double Lane Closure', 'Work Zone + Detour',
                'Night Works (Time-Varying)', 'Moving Work Zone']
k_inflow     = 0.05  

print("Running all scenarios...")
results = {}
for si, sc in enumerate(scenarios):
    print(f"  [{si+1}/{len(scenarios)}] {scen_labels[si]} ...")
    K, V, Q, Wh, t = run_sim(sc, p, k_inflow, save_stride)
    results[sc] = dict(K=K, V=V, Q=Q, Wh=Wh, t=t)
print("All scenarios complete.\n")

t_ref  = results['baseline']['t']
x_km   = p['x'] / 1000.0
ss_idx = round(0.65 * len(t_ref))

#  SECTION 6: STATIC FIELDS

k_range   = np.linspace(p['k_min'], p['k0'], 500)
L_vals    = [1.0, 2.0/3.0, 0.5, 1.0/3.0]
L_labels  = ['L=1 (Baseline)', 'L=2/3 (1 lane closed)',
             'L=1/2 (half closed)', 'L=1/3 (2 lanes closed)']
col4      = plt.cm.tab10(np.linspace(0, 0.4, 4))
col6      = plt.cm.tab10(np.linspace(0, 0.6, 6))

L_stat, W_stat, _, dL_stat = build_construction_fields('single_lane', 0.0, p)
Wh_stat = W_stat.copy()
vf_stat = p['vf0'] * (1.0 - p['etaW'] * W_stat)
Tc_stat = p['T0']  * (1.0 + p['tau_h'] * Wh_stat)   

Running all scenarios...
  [1/6] Baseline (No Construction) ...
  [2/6] Single Lane Closure ...
  [3/6] Double Lane Closure ...
  [4/6] Work Zone + Detour ...
  [5/6] Night Works (Time-Varying) ...
  [6/6] Moving Work Zone ...
All scenarios complete.



In [8]:
#  FIGURE 1: CONSTRUCTION FIELD

fig1, axes1 = plt.subplots(2, 3, figsize=(14, 7))
fig1.canvas.manager.set_window_title('Fig1: Construction Fields')

ax = axes1[0, 0]
ax.plot(x_km, L_stat, 'b-', linewidth=2.5)
ax.axhline(2/3, color='k', linestyle='--', label='L_after=2/3')
ax.set_xlabel('Position (km)'); ax.set_ylabel('Lane fraction L(x,t)')
ax.set_title('Lane Fraction Profile: Single Lane Closure', fontweight='bold')
ax.set_ylim([0, 1.1]); ax.legend(fontsize=9)

ax = axes1[0, 1]
ax.plot(x_km, W_stat, 'r-', linewidth=2.5)
ax.set_xlabel('Position (km)'); ax.set_ylabel('Work activity field W(x,t)')
ax.set_title('Work Activity Profile', fontweight='bold')
ax.set_ylim([0, 1.05])

ax = axes1[0, 2]
ax.plot(x_km, vf_stat, color=[0.1, 0.6, 0.1], linewidth=2.5)
ax.axhline(p['vf0'], color='k', linestyle='--', label='vf0')
ax.set_xlabel('Position (km)'); ax.set_ylabel('Free-flow speed vf')
ax.set_title('Construction-Modified Free-Flow Speed', fontweight='bold')
ax.legend(fontsize=9)

ax = axes1[1, 0]
ax.plot(x_km, Tc_stat, 'm-', linewidth=2.5)
ax.axhline(p['T0'], color='k', linestyle='--', label='T0')
ax.set_xlabel('Position (km)'); ax.set_ylabel('Relaxation time')
ax.set_title('Driver Hesitation', fontweight='bold')
ax.legend(fontsize=9)

ax = axes1[1, 1]
ax.plot(x_km, dL_stat, color=[0.8, 0.4, 0], linewidth=2.5)
ax.axhline(0, color='k', linewidth=0.8)
ax.set_xlabel('Position (km)'); ax.set_ylabel('∂L/∂x')
ax.set_title('Lane Gradient ∂L/∂x', fontweight='bold')

#v_ref_drag    = 15.0
drag_profile  = p['alpha'] * Wh_stat * vf_stat
ax = axes1[1, 2]
ax.plot(x_km, drag_profile, color=[0.5, 0, 0.5], linewidth=2.5)
ax.set_xlabel('Position (km)'); ax.set_ylabel('α·Ŵ·v (free-flow speed)')
ax.set_title('Construction Drag Profile', fontweight='bold')

plt.tight_layout()


In [9]:
#  FIGURE 2: FUNDAMENTAL DIAGRAM

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 5))
fig2.canvas.manager.set_window_title('Fig2: Fundamental Diagrams')

ax = axes2[0]
for li, (Lv, lbl) in enumerate(zip(L_vals, L_labels)):
    keff  = p['k0'] * Lv
    ve_l  = compute_ve(k_range, p['vf0'] * np.ones_like(k_range),
                       keff * np.ones_like(k_range), p['cm'])
    ax.plot(k_range, ve_l, color=col4[li], linewidth=2.2, label=lbl)
ax.set_xlabel('Density k (veh/m)'); ax.set_ylabel('Speed v (m/s)')
ax.set_title('Speed-Density: Lane Fraction Effect', fontweight='bold')
ax.legend(fontsize=9)

ax = axes2[1]
for li, (Lv, lbl) in enumerate(zip(L_vals, L_labels)):
    keff  = p['k0'] * Lv
    ve_l  = compute_ve(k_range, p['vf0'] * np.ones_like(k_range),
                       keff * np.ones_like(k_range), p['cm'])
    ax.plot(k_range, k_range * ve_l, color=col4[li], linewidth=2.2, label=lbl)
ax.set_xlabel('Density k (veh/m)'); ax.set_ylabel('Flow q (veh/s)')
ax.set_title('Flow-Density: Lane Fraction Effect', fontweight='bold')
ax.legend(fontsize=9)

W_vals   = [0.0, 0.3, 0.6, 0.9]
W_labels = [f'W={w:.1f}' for w in W_vals]
ax = axes2[2]
for wi, (wv, lbl) in enumerate(zip(W_vals, W_labels)):
    vf_w = p['vf0'] * (1.0 - p['etaW'] * wv)
    ve_w = compute_ve(k_range, vf_w * np.ones_like(k_range),
                      p['k0'] * np.ones_like(k_range), p['cm'])
    ax.plot(k_range, k_range * ve_w, color=col4[wi], linewidth=2.2, label=lbl)
ax.set_xlabel('Density k (veh/m)'); ax.set_ylabel('Flow q (veh/s)')
ax.set_title('Flow-Density: Work Activity Effect', fontweight='bold')
ax.legend(fontsize=9)

plt.tight_layout()

/tmp/ipykernel_14563/2896213877.py:11: RuntimeWarning: overflow encountered in exp
  return vf * np.maximum(0.0, 1.0 - np.exp(1.0 - np.exp(arg)))


In [10]:
#  FIGURE 3: SPATIO-TEMPORAL FIELDS — BASELINE vs SINGLE LANE

fig3, axes3 = plt.subplots(2, 4, figsize=(16, 8))
fig3.canvas.manager.set_window_title('Fig3: Space-Time Fields')

plot_keys_k = ['baseline', 'single_lane', 'double_lane', 'detour']
plot_titles_k = ['Density: Baseline', 'Density: Single Lane Closure',
                 'Density: Double Lane Closure', 'Density: Work Zone + Detour']
plot_titles_v = ['Speed: Baseline', 'Speed: Single Lane Closure',
                 'Speed: Double Lane Closure', 'Speed: Work Zone + Detour']

for col_i, sc in enumerate(plot_keys_k):
    tv = results[sc]['t']
    im = axes3[0, col_i].imshow(results[sc]['K'], aspect='auto', origin='lower',
                                 extent=[tv[0], tv[-1], x_km[0], x_km[-1]],
                                 cmap='viridis', vmin=0, vmax=p['k0'])
    plt.colorbar(im, ax=axes3[0, col_i])
    axes3[0, col_i].set_xlabel('Time (s)'); axes3[0, col_i].set_ylabel('Position (km)')
    axes3[0, col_i].set_title(plot_titles_k[col_i], fontweight='bold')

    im = axes3[1, col_i].imshow(results[sc]['V'], aspect='auto', origin='lower',
                                 extent=[tv[0], tv[-1], x_km[0], x_km[-1]],
                                 cmap='hot', vmin=0, vmax=p['vf0'])
    plt.colorbar(im, ax=axes3[1, col_i])
    axes3[1, col_i].set_xlabel('Time (s)'); axes3[1, col_i].set_ylabel('Position (km)')
    axes3[1, col_i].set_title(plot_titles_v[col_i], fontweight='bold')

plt.tight_layout()

In [11]:
#  FIGURE 4: EVALUATING SIX SCENARIOS

fig5, axes5 = plt.subplots(2, 2, figsize=(13, 9))
fig5.canvas.manager.set_window_title('Fig5: Midpoint Time Series')

mid_idx = round(p['Nx'] * 1800.0 / p['L_road'])
col_sc  = plt.cm.tab10(np.linspace(0, 0.6, len(scenarios)))

for si, (sc, lbl) in enumerate(zip(scenarios, scen_labels)):
    tv = results[sc]['t']
    axes5[0, 0].plot(tv, results[sc]['K'][mid_idx, :], linewidth=2, color=col_sc[si], label=lbl)
    axes5[0, 1].plot(tv, results[sc]['V'][mid_idx, :], linewidth=2, color=col_sc[si], label=lbl)
    axes5[1, 0].plot(tv, results[sc]['Q'][mid_idx, :], linewidth=2, color=col_sc[si], label=lbl)

axes5[0, 0].set_xlabel('Time (s)'); axes5[0, 0].set_ylabel('Density (veh/m)')
axes5[0, 0].set_title('Density at Work Zone Centre (x≈1.8 km)', fontweight='bold')
axes5[0, 0].legend(fontsize=8)
axes5[0, 1].set_xlabel('Time (s)'); axes5[0, 1].set_ylabel('Speed (m/s)')
axes5[0, 1].set_title('Speed at Work Zone Centre', fontweight='bold')
axes5[0, 1].legend(fontsize=8)
axes5[1, 0].set_xlabel('Time (s)'); axes5[1, 0].set_ylabel('Flow (veh/s)')
axes5[1, 0].set_title('Flow at Work Zone Centre', fontweight='bold')
axes5[1, 0].legend(fontsize=8)

ss_v = np.zeros(len(scenarios))
ss_q = np.zeros(len(scenarios))
for si, sc in enumerate(scenarios):
    Nss    = results[sc]['V'].shape[1]
    idx_ss = round(0.65 * Nss)
    ss_v[si] = results[sc]['V'][:, idx_ss:].mean()
    ss_q[si] = results[sc]['Q'][:, idx_ss:].mean()

ax5b = axes5[1, 1]
x_bar = np.arange(len(scenarios))
bars1 = ax5b.bar(x_bar - 0.2, ss_v, 0.35, color=[0.2, 0.5, 0.8], label='Avg Speed')
ax5b2 = ax5b.twinx()
bars2 = ax5b2.bar(x_bar + 0.2, ss_q, 0.35, color=[0.8, 0.3, 0.3], label='Avg Flow')
ax5b.set_ylabel('Mean Speed (m/s)')
ax5b2.set_ylabel('Mean Flow (veh/s)')
ax5b.set_xticks(x_bar)
ax5b.set_xticklabels(['Base','1-Lane','2-Lane','Detour','Night','Moving'],
                     fontsize=8, rotation=15)
ax5b.set_title('Steady-State Performance Metrics', fontweight='bold')
ax5b.legend(loc='upper left', fontsize=9); ax5b2.legend(loc='upper right', fontsize=9)


plt.tight_layout()

In [12]:
#  FIGURE 5: MOVING WORK ZONE AND NIGHT SHIFT

fig6, axes6 = plt.subplots(2, 3, figsize=(14, 9))
fig6.canvas.manager.set_window_title('Fig6: Time-Varying Scenarios')

def make_xt_plots(axes_row, sc, titles, cmaps, vlims):
    tv = results[sc]['t']
    fields = [results[sc]['K'], results[sc]['V'], results[sc]['Wh']]
    for col_i, (field, title, cmap, vlim) in enumerate(zip(fields, titles, cmaps, vlims)):
        im = axes_row[col_i].imshow(field, aspect='auto', origin='lower',
                                     extent=[tv[0], tv[-1], x_km[0], x_km[-1]],
                                     cmap=cmap,
                                     vmin=vlim[0] if vlim else None,
                                     vmax=vlim[1] if vlim else None)
        plt.colorbar(im, ax=axes_row[col_i])
        axes_row[col_i].set_xlabel('Time (s)'); axes_row[col_i].set_ylabel('Position (km)')
        axes_row[col_i].set_title(title, fontweight='bold')

make_xt_plots(axes6[0], 'moving_zone',
              ['Density: Moving Work Zone', 'Speed: Moving Work Zone', 'Activity Ẇ: Moving Zone'],
              ['viridis', 'hot', 'autumn'],
              [(0, p['k0']), (0, p['vf0']), (0, 1.5)])  
make_xt_plots(axes6[1], 'night_works',
              ['Density: Night Works', 'Speed: Night Works', 'Activity Ẇ: Night Works'],
              ['viridis', 'hot', 'autumn'],
              [(0, p['k0']), (0, p['vf0']), (0, 1.5)])

plt.tight_layout()

In [13]:
#  FIGURE 6: PHASE PORTRAITS

fig7, axes7 = plt.subplots(1, 2, figsize=(10, 5))
fig7.canvas.manager.set_window_title('Fig7: Phase Portraits')

mid_rng  = slice(round(0.35 * p['Nx']), round(0.65 * p['Nx']))

ve_base  = compute_ve(k_range, p['vf0'] * np.ones_like(k_range),
                      p['k0']  * np.ones_like(k_range), p['cm'])
q_base   = k_range * ve_base

sc_plot   = ['baseline', 'single_lane', 'double_lane']
sc_lbl_p  = ['Baseline', 'Single Lane', 'Double Lane']

ax = axes7[0]
ax.plot(k_range, q_base, 'k-', linewidth=2, label='Equil.')
for si, (sc, lbl) in enumerate(zip(sc_plot, sc_lbl_p)):
    k_f = results[sc]['K'][mid_rng, :].ravel()
    q_f = results[sc]['Q'][mid_rng, :].ravel()
    ax.scatter(k_f[::5], q_f[::5], s=5, color=col4[si], alpha=0.4, label=lbl)
ax.set_xlabel('Density k (veh/m)'); ax.set_ylabel('Flow q (veh/s)')
ax.set_title('k-q Portrait: Inside Zone', fontweight='bold'); ax.legend(fontsize=9)

ax = axes7[1]
ax.plot(k_range, ve_base, 'k-', linewidth=2, label='Equil.')
for si, (sc, lbl) in enumerate(zip(sc_plot, sc_lbl_p)):
    k_f = results[sc]['K'][mid_rng, :].ravel()
    v_f = results[sc]['V'][mid_rng, :].ravel()
    ax.scatter(k_f[::5], v_f[::5], s=5, color=col4[si], alpha=0.4, label=lbl)
ax.set_xlabel('Density k (veh/m)'); ax.set_ylabel('Speed v (m/s)')
ax.set_title('v-k Portrait: Inside Zone', fontweight='bold'); ax.legend(fontsize=9)

plt.tight_layout()

In [14]:
#  FIGURE 7: SENSITIVITY ANALYSIS

fig8, axes8 = plt.subplots(2, 2, figsize=(12, 8))
fig8.canvas.manager.set_window_title('Fig8: Sensitivity alpha lambdaL')

alpha_list   = [0, 0.05, 0.10, 0.20, 0.35]
lambdaL_list = [0, 4, 8, 15, 25]
etaW_list    = [0, 0.15, 0.30, 0.45, 0.60]
tau_h_list   = [0, 0.5, 1.0, 1.8, 3.0]

def sweep_param(param_key, param_vals, sc, base_p, k_in, stride):
    avg_v, avg_q = [], []
    p_tmp = dict(base_p)

    for val in param_vals:
        p_tmp[param_key] = val
        _, V_a, Q_a, _, _ = run_sim(sc, p_tmp, k_in, stride)

        idx_ss = round(0.65 * V_a.shape[1])

        avg_v.append(V_a[:, idx_ss:].mean())
        avg_q.append(Q_a[:, idx_ss:].mean())

    return np.array(avg_v), np.array(avg_q)

print("  Sensitivity: alpha sweep...")
avg_v_alpha, avg_q_alpha = sweep_param(
    'alpha', alpha_list, 'single_lane', p, k_inflow, save_stride
)

print("  Sensitivity: lambdaL sweep...")
avg_v_lL, avg_q_lL = sweep_param(
    'lambdaL', lambdaL_list, 'single_lane', p, k_inflow, save_stride
)


# Plot 1
axes8[0, 0].plot(
    alpha_list, avg_v_alpha,
    'bo-', linewidth=2.2, markersize=8
)

axes8[0, 0].set_xlabel('Drag gain α (s⁻¹)')
axes8[0, 0].set_ylabel('Steady-State Mean Speed (m/s)')
axes8[0, 0].set_title('Speed vs. α (Single Lane)', fontweight='bold')


# Plot 2

axes8[0, 1].plot(
    alpha_list, avg_q_alpha,
    'rs-', linewidth=2.2, markersize=8
)

axes8[0, 1].set_xlabel('Drag gain α (s⁻¹)')
axes8[0, 1].set_ylabel('Steady-State Mean Flow (veh/s)')
axes8[0, 1].set_title('Flow vs. α (Single Lane)', fontweight='bold')


# Plot 5
print("  Sensitivity: etaW sweep...")
avg_v_etaW, _ = sweep_param(
    'etaW', etaW_list, 'single_lane', p, k_inflow, save_stride
)

axes8[1, 0].plot(
    etaW_list, avg_v_etaW,
    'c^-', linewidth=2.2, markersize=8
)

axes8[1, 0].set_xlabel('Free-speed sensitivity η_W')
axes8[1, 0].set_ylabel('Steady-State Mean Speed (m/s)')
axes8[1, 0].set_title('Speed vs. η_W', fontweight='bold')


# Plot 6
print("  Sensitivity: tau_h sweep...")
avg_v_tauh, _ = sweep_param(
    'tau_h', tau_h_list, 'single_lane', p, k_inflow, save_stride
)

axes8[1, 1].plot(
    tau_h_list, avg_v_tauh,
    color=[0.6, 0.1, 0.1],
    marker='^',
    linewidth=2.2,
    markersize=8
)

axes8[1, 1].set_xlabel('Hesitation parameter τ')
axes8[1, 1].set_ylabel('Steady-State Mean Speed (m/s)')
axes8[1, 1].set_title('Speed vs. Hesitation τ', fontweight='bold')

plt.tight_layout()

  Sensitivity: alpha sweep...
  Sensitivity: lambdaL sweep...
  Sensitivity: etaW sweep...
  Sensitivity: tau_h sweep...


In [15]:
#  FIGURE 8: DEMAND SENSITIVITY

fig10, axes10 = plt.subplots(2, 3, figsize=(14, 9))
fig10.canvas.manager.set_window_title('Fig10: Demand Sensitivity')

demand_vals = [0.03, 0.05, 0.08, 0.10, 0.12]
d_labels    = [f'k_in={d:.2f}' for d in demand_vals]
col_d       = plt.cm.tab10(np.linspace(0, 0.5, len(demand_vals)))

print("  Demand sensitivity (single_lane)...")
V_demand, K_demand, Q_demand = [], [], []
tv_dem = None
for di, d in enumerate(demand_vals):
    K_d, V_d, Q_d, _, tv_dem = run_sim('single_lane', p, d, save_stride)
    K_demand.append(K_d); V_demand.append(V_d); Q_demand.append(Q_d)

idx_ss_d  = np.argmin(np.abs(tv_dem - 600.0))
mid_zone  = round(p['Nx'] * 1800.0 / p['L_road'])

for di, lbl in enumerate(d_labels):
    axes10[0, 0].plot(x_km, K_demand[di][:, idx_ss_d], color=col_d[di], linewidth=2, label=lbl)
    axes10[0, 1].plot(x_km, V_demand[di][:, idx_ss_d], color=col_d[di], linewidth=2, label=lbl)
    axes10[0, 2].plot(x_km, Q_demand[di][:, idx_ss_d], color=col_d[di], linewidth=2, label=lbl)
    axes10[1, 0].plot(tv_dem, K_demand[di][mid_zone, :], color=col_d[di], linewidth=2, label=lbl)
    axes10[1, 1].plot(tv_dem, V_demand[di][mid_zone, :], color=col_d[di], linewidth=2, label=lbl)

axes10[0, 0].set_title('Density Profile at t=600 s',    fontweight='bold')
axes10[0, 0].set_xlabel('Position (km)'); axes10[0, 0].set_ylabel('Density (veh/m)')
axes10[0, 0].legend(fontsize=9)
axes10[0, 1].set_title('Speed Profile at t=600 s',      fontweight='bold')
axes10[0, 1].set_xlabel('Position (km)'); axes10[0, 1].set_ylabel('Speed (m/s)')
axes10[0, 1].legend(fontsize=9)
axes10[0, 2].set_title('Flow Profile at t=600 s',       fontweight='bold')
axes10[0, 2].set_xlabel('Position (km)'); axes10[0, 2].set_ylabel('Flow (veh/s)')
axes10[0, 2].legend(fontsize=9)
axes10[1, 0].set_title('Zone Centre Density vs. Time',  fontweight='bold')
axes10[1, 0].set_xlabel('Time (s)'); axes10[1, 0].set_ylabel('Density (veh/m)')
axes10[1, 0].legend(fontsize=9)
axes10[1, 1].set_title('Zone Centre Speed vs. Time',    fontweight='bold')
axes10[1, 1].set_xlabel('Time (s)'); axes10[1, 1].set_ylabel('Speed (m/s)')
axes10[1, 1].legend(fontsize=9)


ss_qu  = round(0.65 * len(tv_dem))
q_len  = []
for di in range(len(demand_vals)):
    k_ss_mat  = K_demand[di][:, ss_qu:]
    frac_cong = np.mean(k_ss_mat > 0.5 * p['k0'])
    q_len.append(frac_cong * p['L_road'] / 1000.0)

axes10[1, 2].bar(np.arange(len(demand_vals)), q_len, color=[0.3, 0.6, 0.9])
axes10[1, 2].set_xticks(np.arange(len(demand_vals)))
axes10[1, 2].set_xticklabels([f'{d:.2f}' for d in demand_vals])
axes10[1, 2].set_xlabel('Inflow Density k_in (veh/m)')
axes10[1, 2].set_ylabel('Congested Corridor Fraction (km)')
axes10[1, 2].set_title('Queue Extent vs. Demand', fontweight='bold')


plt.tight_layout()

  Demand sensitivity (single_lane)...


In [16]:
#  A SUMMARY TABLE

print("\n=====================================================")
print("  STEADY-STATE PERFORMANCE SUMMARY")
print("=====================================================")
print(f"{'Scenario':<26}  {'Mean Speed':>12}  {'Mean Flow':>12}")
print("-" * 54)
for sc, lbl in zip(scenarios, scen_labels):
    Nss   = results[sc]['V'].shape[1]
    idx_s = round(0.65 * Nss)
    mean_v = results[sc]['V'][:, idx_s:].mean()
    mean_q = results[sc]['Q'][:, idx_s:].mean()
    print(f"{lbl:<26}  {mean_v:>12.4f}  {mean_q:>12.4f}")
print("=====================================================")
print("\nAll figures generated. Simulation complete.")

plt.show()


  STEADY-STATE PERFORMANCE SUMMARY
Scenario                      Mean Speed     Mean Flow
------------------------------------------------------
Baseline (No Construction)       17.9857        0.8993
Single Lane Closure              20.2269        0.5569
Double Lane Closure              19.0713        0.4375
Work Zone + Detour               19.3888        0.4980
Night Works (Time-Varying)       22.3164        0.6634
Moving Work Zone                 19.7264        0.5888

All figures generated. Simulation complete.
